# Control Variables - 3.99 Mega merge of control variables

Kuba Kowalski

Last modified on 10/07/2026

This is a combination of all separate scripts written for the control variables 1-16. The variables are first merged to the GEOJSON and then re-ordered and re-named as needed for maximum convenience. Finally, the merged files are exported as GEOJSON, CSV, and Excel. 

In [45]:
# 99 Merge demographic data with all control variables

import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

base_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson"
)

control_files = [
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\1_distance_to_nearest_city\1_distance_to_nearest_city_1900.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation\2_elevation_sd_worldclim.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain\3_average_annual_precipitation_worldclim.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature\4_average_annual_temperature_worldclim.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\6_river\6_main_river_dummy.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\7_sea\7_sea_dummy.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\8_border\8_border_dummy_combined.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\9_lat_lon_area\9_10_11_latitude_longitude_area.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\11_caloric_density_index\11_province_csi_mean_sd_gini.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\12_oil\12_oil_surface_share.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\13_gold\13_gold_deposit_dummy.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\zonal_stats_2000.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\zonal_stats_2012.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\15_railways\15_railway_dummy.csv",
    
    # Missionary-station controls
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group1_mission_dummy_distance.csv",
    #r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group2_mission_staff.csv", # Uncomment if you want staff counts for Protestant missions only
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group3_mission_25km_exposure_buffer.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group4_mission_counts.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16k_mission_station_density.csv",
]

output_geojson = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_final.geojson"
)

output_csv = output_geojson.with_suffix(".csv")
output_excel = output_geojson.with_suffix(".xlsx")

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

join_key = "GEOLEVEL1"

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------

def clean_geolevel1(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x.zfill(6)

# ------------------------------------------------------------------
# LOAD BASE FILE
# ------------------------------------------------------------------

master = gpd.read_file(base_file)

if join_key not in master.columns:
    raise ValueError(f"Join key '{join_key}' not found in base file.")

master[join_key] = master[join_key].apply(clean_geolevel1)

base_row_count = len(master)
base_cols = set(master.columns)

print("Base rows:", len(master))
print("Base columns:", len(master.columns))

# ------------------------------------------------------------------
# MERGE CONTROL FILES
# ------------------------------------------------------------------

for control_path in control_files:

    control_path = Path(control_path)
    print(f"\nMerging: {control_path.name}")

    if not control_path.exists():
        raise FileNotFoundError(f"Missing control file: {control_path}")

    control = pd.read_csv(control_path, dtype={join_key: str})

    if join_key not in control.columns:
        raise ValueError(f"Join key '{join_key}' not found in: {control_path}")

    control[join_key] = control[join_key].apply(clean_geolevel1)

    drop_cols = [
        col for col in control.columns
        if col.lower() in ["geometry", "geom", "wkt"]
    ]

    if drop_cols:
        control = control.drop(columns=drop_cols)

    duplicate_count = control[join_key].duplicated().sum()

    if duplicate_count > 0:
        print(f"Warning: {duplicate_count} duplicate {join_key} rows found. Keeping first.")
        control = control.drop_duplicates(subset=[join_key], keep="first")

    duplicate_cols = [
        col for col in control.columns
        if col != join_key and col in master.columns
    ]

    if duplicate_cols:
        print(f"Dropping already-existing columns: {duplicate_cols}")
        control = control.drop(columns=duplicate_cols)

    before_cols = len(master.columns)

    master = master.merge(
        control,
        on=join_key,
        how="left",
        validate="many_to_one"
    )

    added_cols = len(master.columns) - before_cols

    print(f"Added columns: {added_cols}")
    print(f"Rows after merge: {len(master)}")

    if len(master) != base_row_count:
        raise ValueError(f"Row count changed after merging {control_path.name}")

# ------------------------------------------------------------------
# CLEAN FINAL OUTPUT COLUMNS BEFORE EXPORT
# ------------------------------------------------------------------

rename_columns = {
    "csi_mean": "11_caloric_suitability_average",
    "mean_2000": "14a_nighttime_light_intensity_average_2000",
    "mean_2012": "14b_nighttime_light_intensity_average_2012",
    "11_area-km2": "5_area-km-2",
    "8_border-dummy": "8_border",
    "3_average-annual-precipitation-mm":"3a_average-annual-precipitation-mm",
    "2_elevation-sd":"2a_elevation-sd",
    "4_average-annual-temperature-c":"4a_average-annual-temperature-c",
}

drop_columns = [
    "year",
    "count",
    "4",
    "min",
    "8a_border-with-selected-country",
    "8b_border-with-nonselected-country",
    "max",
    "std",
    "sum",
    "median",
    "range",
    "csi_sd",
    "csi_gini",
    "csi_cell_count",
    "16b_nearest-mission-id",
    "4e_annual-temperature-valid-pixels",
]

# Exact-match removal only
existing_drop_columns = [
    col for col in drop_columns
    if col in master.columns
]

if existing_drop_columns:
    print("Dropping columns:")
    print(existing_drop_columns)
    master = master.drop(columns=existing_drop_columns)

# Exact-match renaming only
existing_rename_columns = {
    old: new
    for old, new in rename_columns.items()
    if old in master.columns
}

if existing_rename_columns:
    print("Renaming columns:")
    print(existing_rename_columns)
    master = master.rename(columns=existing_rename_columns)

# ------------------------------------------------------------------
# REORDER FINAL OUTPUT COLUMNS
# ------------------------------------------------------------------

column_to_move = "5_area-km-2"
insert_after = "4d_annual-temperature-sd-c"

if column_to_move in master.columns and insert_after in master.columns:
    cols = master.columns.tolist()

    cols.remove(column_to_move)

    insert_position = cols.index(insert_after) + 1
    cols.insert(insert_position, column_to_move)

    master = master[cols]
else:
    print(
        f"Could not reorder columns. "
        f"Found {column_to_move}: {column_to_move in master.columns}; "
        f"found {insert_after}: {insert_after in master.columns}"
    )

# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

output_geojson.parent.mkdir(parents=True, exist_ok=True)

master.to_file(output_geojson, driver="GeoJSON")
print(f"\nGeoJSON saved to: {output_geojson}")

master.drop(columns="geometry").to_csv(output_csv, index=False)
print(f"CSV saved to: {output_csv}")

master.drop(columns="geometry").to_excel(
    output_excel,
    index=False,
    engine="openpyxl"
)
print(f"Excel saved to: {output_excel}")

print("Done.")

Base rows: 2045
Base columns: 10

Merging: 1_distance_to_nearest_city_1900.csv
Added columns: 2
Rows after merge: 2045

Merging: 2_elevation_sd_worldclim.csv
Added columns: 5
Rows after merge: 2045

Merging: 3_average_annual_precipitation_worldclim.csv
Added columns: 5
Rows after merge: 2045

Merging: 4_average_annual_temperature_worldclim.csv
Added columns: 5
Rows after merge: 2045

Merging: 6_main_river_dummy.csv
Added columns: 1
Rows after merge: 2045

Merging: 7_sea_dummy.csv
Added columns: 1
Rows after merge: 2045

Merging: 8_border_dummy_combined.csv
Dropping already-existing columns: ['country']
Added columns: 3
Rows after merge: 2045

Merging: 9_10_11_latitude_longitude_area.csv
Added columns: 3
Rows after merge: 2045

Merging: 11_province_csi_mean_sd_gini.csv
Added columns: 4
Rows after merge: 2045

Merging: 12_oil_surface_share.csv
Added columns: 1
Rows after merge: 2045

Merging: 13_gold_deposit_dummy.csv
Added columns: 1
Rows after merge: 2045

Merging: zonal_stats_2000.csv

In [46]:
# 99 Merge demographic data with all control variables

import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

base_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

control_files = [
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\1_distance_to_nearest_city\1_distance_to_nearest_city_1900.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation\2_elevation_sd_worldclim.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain\3_average_annual_precipitation_worldclim.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature\4_average_annual_temperature_worldclim.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\6_river\6_main_river_dummy.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\7_sea\7_sea_dummy.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\8_border\8_border_dummy_combined.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\9_lat_lon_area\9_10_11_latitude_longitude_area.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\11_caloric_density_index\11_province_csi_mean_sd_gini.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\12_oil\12_oil_surface_share.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\13_gold\13_gold_deposit_dummy.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\zonal_stats_2000.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\zonal_stats_2012.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\15_railways\15_railway_dummy.csv",
    
    # Missionary-station controls
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group1_mission_dummy_distance.csv",
    #r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group2_mission_staff.csv", # Uncomment if you want staff counts for Protestant missions only
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group3_mission_25km_exposure_buffer.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group4_mission_counts.csv",
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16k_mission_station_density.csv",
]

output_geojson = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_final.geojson"
)

output_csv = output_geojson.with_suffix(".csv")
output_excel = output_geojson.with_suffix(".xlsx")

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

join_key = "GEOLEVEL1"

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------

def clean_geolevel1(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x.zfill(6)

# ------------------------------------------------------------------
# LOAD BASE FILE
# ------------------------------------------------------------------

master = gpd.read_file(base_file)

if join_key not in master.columns:
    raise ValueError(f"Join key '{join_key}' not found in base file.")

master[join_key] = master[join_key].apply(clean_geolevel1)

base_row_count = len(master)
base_cols = set(master.columns)

print("Base rows:", len(master))
print("Base columns:", len(master.columns))

# ------------------------------------------------------------------
# MERGE CONTROL FILES
# ------------------------------------------------------------------

for control_path in control_files:

    control_path = Path(control_path)
    print(f"\nMerging: {control_path.name}")

    if not control_path.exists():
        raise FileNotFoundError(f"Missing control file: {control_path}")

    control = pd.read_csv(control_path, dtype={join_key: str})

    if join_key not in control.columns:
        raise ValueError(f"Join key '{join_key}' not found in: {control_path}")

    control[join_key] = control[join_key].apply(clean_geolevel1)

    drop_cols = [
        col for col in control.columns
        if col.lower() in ["geometry", "geom", "wkt"]
    ]

    if drop_cols:
        control = control.drop(columns=drop_cols)

    duplicate_count = control[join_key].duplicated().sum()

    if duplicate_count > 0:
        print(f"Warning: {duplicate_count} duplicate {join_key} rows found. Keeping first.")
        control = control.drop_duplicates(subset=[join_key], keep="first")

    duplicate_cols = [
        col for col in control.columns
        if col != join_key and col in master.columns
    ]

    if duplicate_cols:
        print(f"Dropping already-existing columns: {duplicate_cols}")
        control = control.drop(columns=duplicate_cols)

    before_cols = len(master.columns)

    master = master.merge(
        control,
        on=join_key,
        how="left",
        validate="many_to_one"
    )

    added_cols = len(master.columns) - before_cols

    print(f"Added columns: {added_cols}")
    print(f"Rows after merge: {len(master)}")

    if len(master) != base_row_count:
        raise ValueError(f"Row count changed after merging {control_path.name}")

# ------------------------------------------------------------------
# CLEAN FINAL OUTPUT COLUMNS BEFORE EXPORT
# ------------------------------------------------------------------

rename_columns = {
    "csi_mean": "11_caloric_suitability_average",
    "mean": "14_nighttime_light_intensity_average_2000",
    "11_area-km2": "5_area-km-2",
    "8_border-dummy": "8_border",
    "3_average-annual-precipitation-mm":"3a_average-annual-precipitation-mm",
    "2_elevation-sd":"2a_elevation-sd",
    "4_average-annual-temperature-c":"4a_average-annual-temperature-c",
}

drop_columns = [
    "year",
    "count",
    "4",
    "min",
    "8a_border-with-selected-country",
    "8b_border-with-nonselected-country",
    "max",
    "std",
    "sum",
    "median",
    "range",
    "csi_sd",
    "csi_gini",
    "csi_cell_count",
    "16b_nearest-mission-id",
    "4e_annual-temperature-valid-pixels",
]

# Exact-match removal only
existing_drop_columns = [
    col for col in drop_columns
    if col in master.columns
]

if existing_drop_columns:
    print("Dropping columns:")
    print(existing_drop_columns)
    master = master.drop(columns=existing_drop_columns)

# Exact-match renaming only
existing_rename_columns = {
    old: new
    for old, new in rename_columns.items()
    if old in master.columns
}

if existing_rename_columns:
    print("Renaming columns:")
    print(existing_rename_columns)
    master = master.rename(columns=existing_rename_columns)

# ------------------------------------------------------------------
# REORDER FINAL OUTPUT COLUMNS
# ------------------------------------------------------------------

column_to_move = "5_area-km-2"
insert_after = "4d_annual-temperature-sd-c"

if column_to_move in master.columns and insert_after in master.columns:
    cols = master.columns.tolist()

    cols.remove(column_to_move)

    insert_position = cols.index(insert_after) + 1
    cols.insert(insert_position, column_to_move)

    master = master[cols]
else:
    print(
        f"Could not reorder columns. "
        f"Found {column_to_move}: {column_to_move in master.columns}; "
        f"found {insert_after}: {insert_after in master.columns}"
    )

# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

output_geojson.parent.mkdir(parents=True, exist_ok=True)

master.to_file(output_geojson, driver="GeoJSON")
print(f"\nGeoJSON saved to: {output_geojson}")

master.drop(columns="geometry").to_csv(output_csv, index=False)
print(f"CSV saved to: {output_csv}")

master.drop(columns="geometry").to_excel(
    output_excel,
    index=False,
    engine="openpyxl"
)
print(f"Excel saved to: {output_excel}")

print("Done.")

Base rows: 2125
Base columns: 10

Merging: 1_distance_to_nearest_city_1900.csv
Added columns: 2
Rows after merge: 2125

Merging: 2_elevation_sd_worldclim.csv
Added columns: 5
Rows after merge: 2125

Merging: 3_average_annual_precipitation_worldclim.csv
Added columns: 5
Rows after merge: 2125

Merging: 4_average_annual_temperature_worldclim.csv
Added columns: 5
Rows after merge: 2125

Merging: 6_main_river_dummy.csv
Added columns: 1
Rows after merge: 2125

Merging: 7_sea_dummy.csv
Added columns: 1
Rows after merge: 2125

Merging: 8_border_dummy_combined.csv
Dropping already-existing columns: ['country']
Added columns: 3
Rows after merge: 2125

Merging: 9_10_11_latitude_longitude_area.csv
Added columns: 3
Rows after merge: 2125

Merging: 11_province_csi_mean_sd_gini.csv
Added columns: 4
Rows after merge: 2125

Merging: 12_oil_surface_share.csv
Added columns: 1
Rows after merge: 2125

Merging: 13_gold_deposit_dummy.csv
Added columns: 1
Rows after merge: 2125

Merging: zonal_stats_2000.csv

## Diagnostics of the final results for documentation & meetings

Cells below are irrelevant for any of the results and are merely diagnostics. 

In [47]:
from pathlib import Path
import geopandas as gpd

output_geojson = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_final.geojson"
)

gdf = gpd.read_file(output_geojson)

# Inspect column names first
print(gdf.columns.tolist())

province_counts = (
    gdf.groupby("cohort")["GEOLEVEL1"]
       .nunique()
       .reset_index(name="n_provinces")
       .sort_values("cohort")
)

print(province_counts)

['country_id', 'country', 'GEOLEVEL1', 'province', 'cohort', 'primary_educ', 'higher_educ', 'tertiary_educ', 'n', '1a_distance-to-the-nearest-city', '1b_name-of-the-nearest-city', '2a_elevation-sd', '2b_elevation-mean', '2c_elevation-min', '2d_elevation-max', '2e_elevation-valid-pixels', '3a_average-annual-precipitation-mm', '3b_annual-precipitation-min-mm', '3c_annual-precipitation-max-mm', '3d_annual-precipitation-sd-mm', '3e_annual-precipitation-valid-pixels', '4a_average-annual-temperature-c', '4b_annual-temperature-min-c', '4c_annual-temperature-max-c', '4d_annual-temperature-sd-c', '5_area-km-2', '6_river', '7_sea', '8_border', '9_latitude', '10_longitude', '11_caloric_suitability_average', '12_oil-surface-share', '13_gold-deposit', '14a_nighttime_light_intensity_average_2000', '14b_nighttime_light_intensity_average_2012', '15_railway', '16a_mission-dummy', '16b_distance-to-nearest-mission-km', '16e_25-km-mission-exposure-buffer', '16f_mission-count-within-25km-centroid', '16g_mi

In [48]:
from pathlib import Path
import geopandas as gpd
from functools import reduce

output_geojson = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_final.geojson"
)

gdf = gpd.read_file(output_geojson)

# Change this if your province column has a different name
province_col = "GEOLEVEL1"

province_sets = (
    gdf.groupby("cohort")[province_col]
       .apply(lambda x: set(x.dropna()))
       .to_dict()
)

common_provinces = reduce(set.intersection, province_sets.values())
all_provinces = reduce(set.union, province_sets.values())

problem_provinces = sorted(all_provinces - common_provinces)

print(f"Provinces not present in every cohort: {len(problem_provinces)}")
print(problem_provinces)

for cohort in sorted(province_sets):
    missing = sorted(all_provinces - province_sets[cohort])
    extra = sorted(province_sets[cohort] - common_provinces)

    print(f"\nCohort {cohort}")
    print(f"n_provinces: {len(province_sets[cohort])}")
    print(f"Missing from this cohort: {missing}")

Provinces not present in every cohort: 12
['072001', '072004', '716000', '716001', '716002', '716003', '716004', '716005', '716006', '716007', '716008', '716009']

Cohort 1915
n_provinces: 282
Missing from this cohort: ['072001', '072004', '716000', '716001', '716002', '716003', '716004', '716005', '716006', '716007', '716008', '716009']

Cohort 1925
n_provinces: 294
Missing from this cohort: []

Cohort 1935
n_provinces: 293
Missing from this cohort: ['072004']

Cohort 1945
n_provinces: 294
Missing from this cohort: []

Cohort 1955
n_provinces: 294
Missing from this cohort: []

Cohort 1965
n_provinces: 294
Missing from this cohort: []

Cohort 1975
n_provinces: 294
Missing from this cohort: []


In [49]:
from pathlib import Path
import geopandas as gpd

output_geojson = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_final.geojson"
)

gdf = gpd.read_file(output_geojson)

cohort_sums = (
    gdf.groupby("cohort")["n"]
       .sum()
       .reset_index(name="sum_n")
       .sort_values("cohort")
)

print(cohort_sums)

   cohort    sum_n
0    1915   373473
1    1925   905812
2    1935  1771835
3    1945  2799983
4    1955  4240352
5    1965  6028340
6    1975  7032937


In [50]:
import pandas as pd
from pathlib import Path

target_ids = ["072004", "204008"]

control_files = {
    "elevation": r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation\2_elevation_sd_worldclim.csv",
    "rain": r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\3_rain\3_average_annual_precipitation_worldclim.csv",
    "temperature": r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\4_temperature\4_average_annual_temperature_worldclim.csv",
}

for name, path in control_files.items():
    df = pd.read_csv(path, dtype={"GEOLEVEL1": str})
    df["GEOLEVEL1"] = df["GEOLEVEL1"].str.zfill(6)

    print(f"\n{name}")
    print(df[df["GEOLEVEL1"].isin(target_ids)])


elevation
   GEOLEVEL1  2_elevation-sd  2b_elevation-mean  2c_elevation-min  \
8     072004       31.535694              870.0             830.0   
40    204008        0.000000                4.0               4.0   

    2d_elevation-max  2e_elevation-valid-pixels  
8              917.0                          4  
40               4.0                          1  

rain
   GEOLEVEL1  3_average-annual-precipitation-mm  \
13    204008                             1181.0   
19    204008                             1181.0   
31    204008                             1181.0   
43    204008                             1181.0   
55    204008                             1181.0   
67    204008                             1181.0   
79    204008                             1181.0   
84    072004                              377.5   
85    072004                              377.5   
86    072004                              377.5   
87    072004                              377.5   
88    072004 

In [51]:
gdf = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_final.geojson"
)

print(
    gdf.groupby("cohort")
       .size()
       .sort_index()
)

cohort
1915    295
1925    305
1935    305
1945    305
1955    305
1965    305
1975    305
dtype: int64


In [52]:
import geopandas as gpd

# Load file
gdf = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_final.geojson"
)

# Print all variables with their index
print(f"Total variables: {len(gdf.columns)}\n")

for i, col in enumerate(gdf.columns, start=1):
    print(f"{i:>2}. {col}")

Total variables: 47

 1. country_id
 2. country
 3. GEOLEVEL1
 4. province
 5. cohort
 6. primary_educ
 7. higher_educ
 8. tertiary_educ
 9. n
10. 1a_distance-to-the-nearest-city
11. 1b_name-of-the-nearest-city
12. 2a_elevation-sd
13. 2b_elevation-mean
14. 2c_elevation-min
15. 2d_elevation-max
16. 2e_elevation-valid-pixels
17. 3a_average-annual-precipitation-mm
18. 3b_annual-precipitation-min-mm
19. 3c_annual-precipitation-max-mm
20. 3d_annual-precipitation-sd-mm
21. 3e_annual-precipitation-valid-pixels
22. 4a_average-annual-temperature-c
23. 4b_annual-temperature-min-c
24. 4c_annual-temperature-max-c
25. 4d_annual-temperature-sd-c
26. 5_area-km-2
27. 6_river
28. 7_sea
29. 8_border
30. 9_latitude
31. 10_longitude
32. 11_caloric_suitability_average
33. 12_oil-surface-share
34. 13_gold-deposit
35. 14a_nighttime_light_intensity_average_2000
36. 14b_nighttime_light_intensity_average_2012
37. 15_railway
38. 16a_mission-dummy
39. 16b_distance-to-nearest-mission-km
40. 16e_25-km-mission-expos

In [53]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

input_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_final.geojson"
)

output_file = input_file.with_name("birthplace_final_codebook.txt")

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

# Variables with no more than this number of unique values
# will have their observed values and frequencies printed.
categorical_threshold = 30

# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

gdf = gpd.read_file(input_file)

# Exclude geometry from the codebook
df = gdf.drop(columns="geometry", errors="ignore")

# ------------------------------------------------------------------
# BUILD CODEBOOK
# ------------------------------------------------------------------

lines = []

lines.append(f"DATA CODEBOOK: {input_file.name}")
lines.append(f"Number of rows: {len(df):,}")
lines.append(f"Number of variables: {len(df.columns):,}")
lines.append("=" * 72)
lines.append("")

for number, variable in enumerate(df.columns, start=1):

    series = df[variable]
    non_missing = int(series.notna().sum())
    missing = int(series.isna().sum())
    unique_count = int(series.nunique(dropna=True))

    lines.append(f"{number}. {variable}")
    lines.append("")
    lines.append(f"Type: {series.dtype}")
    lines.append(f"Non-missing values: {non_missing:,}")
    lines.append(f"Missing values: {missing:,}")
    lines.append(f"Unique values: {unique_count:,}")

    # --------------------------------------------------------------
    # NUMERIC SUMMARY
    # --------------------------------------------------------------

    if pd.api.types.is_numeric_dtype(series):

        numeric = pd.to_numeric(series, errors="coerce").dropna()

        if len(numeric) > 0:
            lines.append("")
            lines.append("Summary statistics")
            lines.append(f"Minimum: {numeric.min()}")
            lines.append(f"Maximum: {numeric.max()}")
            lines.append(f"Mean: {numeric.mean()}")
            lines.append(f"Standard deviation: {numeric.std()}")

    # --------------------------------------------------------------
    # OBSERVED VALUE LABELS / FREQUENCIES
    # --------------------------------------------------------------

    if 0 < unique_count <= categorical_threshold:

        counts = (
            series
            .value_counts(dropna=False)
            .rename_axis("Value")
            .reset_index(name="Count")
        )

        lines.append("")
        lines.append("Value\tCount")

        for _, row in counts.iterrows():

            value = row["Value"]

            if pd.isna(value):
                value = "<missing>"

            lines.append(f"{value}\t{int(row['Count'])}")

    lines.append("")
    lines.append("_" * 72)
    lines.append("")

# ------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------

output_file.write_text(
    "\n".join(lines),
    encoding="utf-8"
)

print(f"Codebook saved to:\n{output_file}")

Codebook saved to:
C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\birthplace_final_codebook.txt
